# M1 — TinyStories and custom byte-level BPE
This notebook coordinates repository code. The BPE implementation stays in `src`.


In [ ]:
REPO_URL = "https://github.com/shinbkb/mini-gpt.git"
COMMIT_SHA = "REPLACE_WITH_FULL_COMMIT_CONTAINING_M1"
DATASET_REVISION = None  # Prefer a pinned Hugging Face commit
OFFLINE_SOURCE = None   # Kaggle Input path containing this repository
SOURCE_COMMIT = None    # Required for offline source
TRAIN_FILE = None       # Local JSONL/TXT train split if offline
VALIDATION_FILE = None  # Separate validation file


In [ ]:
from pathlib import Path
import os, re, shutil, subprocess, sys
root = Path('/kaggle/working/mini-gpt')
if OFFLINE_SOURCE:
    assert SOURCE_COMMIT and re.fullmatch(r'[0-9a-fA-F]{40}', SOURCE_COMMIT)
    if not root.exists():
        shutil.copytree(OFFLINE_SOURCE, root, ignore=shutil.ignore_patterns('.venv', '__pycache__'))
else:
    assert re.fullmatch(r'[0-9a-fA-F]{40}', COMMIT_SHA)
    if not root.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(root)], check=True)
    dirty = subprocess.check_output(['git','status','--porcelain'], cwd=root, text=True)
    if dirty.strip(): raise RuntimeError('Checkout has local changes; preserve them before checkout.')
    subprocess.run(['git','fetch','origin',COMMIT_SHA], cwd=root, check=True)
    subprocess.run(['git','checkout','--detach',COMMIT_SHA], cwd=root, check=True)
    actual = subprocess.check_output(['git','rev-parse','HEAD'], cwd=root, text=True).strip()
    assert actual.lower() == COMMIT_SHA.lower()
os.chdir(root)
if not OFFLINE_SOURCE:
    subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'], check=True)
elif not (TRAIN_FILE and VALIDATION_FILE):
    raise ValueError('Offline mode needs separate train and validation files.')


In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q'], check=True)


## Prepare tokenizer and bounded token caches
Default sample: first 20,000 train stories and 1,000 validation stories. The tokenizer is fitted on train only.
For reproducibility, set a dataset revision. Offline mode reads two distinct local files.


In [ ]:
command = [sys.executable,'-m','src.data.prepare','--train-limit','20000',
           '--validation-limit','1000','--vocab-size','8000','--block-size','512',
           '--require-full-vocab']
if TRAIN_FILE and VALIDATION_FILE:
    command += ['--train-file',TRAIN_FILE,'--validation-file',VALIDATION_FILE]
else:
    if not DATASET_REVISION:
        raise ValueError('Set DATASET_REVISION to pin the online dataset version.')
    command += ['--revision',DATASET_REVISION]
subprocess.run(command, check=True)


In [ ]:
from src.tokenizer.bpe import ByteLevelBPE
tokenizer = ByteLevelBPE.load('artifacts/tokenizer.json')
assert tokenizer.vocab_size == 8000
sample = 'Tiếng Việt, emoji 🙂 and punctuation!\n'
assert tokenizer.decode(tokenizer.encode(sample)) == sample
print('Tokenizer round-trip OK; save/download /kaggle/working/mini-gpt/artifacts from Notebook Output.')
